# 01. Exploratory Data Analysis (EDA)
## Intelligent Healthcare Disease Risk Prediction & Clinical Decision Support System

**Domain:** Healthcare / Supervised Learning (Binary Classification)  
**Dataset:** Pima Indians Diabetes Database (`data/diabetes_data.csv`)  
**Target:** `Outcome` (0 = Negative / Lower Risk, 1 = Positive / Higher Risk)  

### Objectives of this Notebook:
1. Inspect data structure, data types, nulls, and duplicate records.
2. Assess target class balance and implications for model evaluation.
3. Identify and quantify the known physiological quirk (zeros used as missing value placeholders).
4. Analyze univariate distributions and skewness.
5. Perform bivariate comparisons (feature values across diabetic vs non-diabetic cohorts).
6. Quantify statistical outliers across all numerical features using the IQR rule.
7. Analyze correlation patterns with the target and multicollinearity between features.
8. Summarize actionable insights for Phase 2 Preprocessing & Feature Engineering.

## 1. Environment Setup & Data Loading

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Visual display settings
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# Load dataset
DATA_PATH = '../data/diabetes_data.csv'
df = pd.read_csv(DATA_PATH)
print('Dataset loaded successfully.')
print(f'Shape: {df.shape}')
df.head()

Dataset loaded successfully.
Shape: (768, 9)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


## 2. Dataset Structure, Data Types & Statistical Summary

In [2]:
df.info()
print('\nExplicit nulls:')
print(df.isnull().sum())
print(f'\nDuplicate rows: {df.duplicated().sum()}')
df.describe()

<class 'pandas.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB

Explicit nulls:
Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                     

       Pregnancies     Glucose  BloodPressure  SkinThickness     Insulin         BMI  DiabetesPedigreeFunction         Age     Outcome
count   768.000000  768.000000     768.000000     768.000000  768.000000  768.000000                768.000000  768.000000  768.000000
mean      3.845052  120.894531      69.105469      20.536458   79.799479   31.992578                  0.471876   33.240885    0.348958
std       3.369578   31.972618      19.355807      15.952218  115.244002    7.884160                  0.331329   11.760232    0.476951
min       0.000000    0.000000       0.000000       0.000000    0.000000    0.000000                  0.078000   21.000000    0.000000
25%       1.000000   99.000000      62.000000       0.000000    0.000000   27.300000                  0.243750   24.000000    0.000000
50%       3.000000  117.000000      72.000000      23.000000   30.500000   32.000000                  0.372500   29.000000    0.000000
75%       6.000000  140.250000      80.000000      32.0

## 3. Target Class Balance (`Outcome`)

In [3]:
counts = df['Outcome'].value_counts()
pcts = (df['Outcome'].value_counts(normalize=True) * 100).round(2)
class_dist = pd.DataFrame({'Count': counts, 'Percentage (%)': pcts})
print('Target Distribution:')
print(class_dist)
ratio = counts[0] / counts[1]
print(f'\nImbalance Ratio (Class 0 : Class 1) = {ratio:.2f} : 1')

Target Distribution:
         Count  Percentage (%)
Outcome                       
0          500           65.10
1          268           34.90

Imbalance Ratio (Class 0 : Class 1) = 1.87 : 1


## 4. Known Data Quirk: Zero-as-Missing Sentinel Values

Notice in `.describe()` that `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, and `BMI` have minimum values of `0.0`. In living human subjects:
- Blood glucose cannot be 0
- Diastolic blood pressure cannot be 0
- Triceps skin fold cannot be 0
- Serum insulin cannot be 0
- BMI cannot be 0

These zeros are unrecorded/missing measurements entered as `0` instead of `NaN`.

In [4]:
quirk_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
zero_records = []
for col in quirk_cols:
    z_count = (df[col] == 0).sum()
    z_pct = round((z_count / len(df)) * 100, 2)
    zero_records.append({'Column': col, 'Zero Count': z_count, 'Percentage (%)': z_pct})

zero_df = pd.DataFrame(zero_records)
print('Features with Zero-as-Missing Sentinel Values:')
print(zero_df.to_string(index=False))

Features with Zero-as-Missing Sentinel Values:
         Column  Zero Count  Percentage (%)
        Glucose           5            0.65
  BloodPressure          35            4.56
  SkinThickness         227           29.56
        Insulin         374           48.70
            BMI          11            1.43


### Clinical Impact of Treating Zeros as Missing:
When zeros are treated as `NaN` rather than true zeros, observe how correlations with `Outcome` shift dramatically (particularly for `Insulin` and `SkinThickness`):

In [5]:
df_nan = df.copy()
for col in quirk_cols:
    df_nan[col] = df_nan[col].replace(0, np.nan)

corr_compare = pd.DataFrame({
    'Raw (with zeros)': df.corr()['Outcome'][quirk_cols],
    'True (zeros as NaN)': df_nan.corr()['Outcome'][quirk_cols]
})
print('Correlation with Outcome:')
print(corr_compare.sort_values(by='True (zeros as NaN)', ascending=False))

Correlation with Outcome:
                  Raw (with zeros)  True (zeros as NaN)
Glucose                   0.466581             0.494650
BMI                       0.292695             0.313680
Insulin                   0.130548             0.303454
SkinThickness             0.074752             0.259491
BloodPressure             0.065068             0.170589


## 5. Univariate Distribution & Skewness Analysis

In [6]:
skewness = df.drop(columns=['Outcome']).skew()
print('Feature Skewness Coefficients:')
for feat, val in skewness.items():
    desc = 'Approximately symmetric' if abs(val) < 0.5 else ('Moderately skewed' if abs(val) < 1.0 else 'Highly skewed')
    direction = 'right-skewed' if val > 0 else 'left-skewed'
    print(f'{feat}: {val:.2f} ({desc} {direction})')

Feature Skewness Coefficients:
Pregnancies: 0.90 (Moderately right-skewed)
Glucose: 0.17 (Approximately symmetric)
BloodPressure: -1.84 (Highly left-skewed due to 0 placeholders)
SkinThickness: 0.11 (Approximately symmetric)
Insulin: 2.27 (Highly right-skewed)
BMI: -0.43 (Approximately symmetric)
DiabetesPedigreeFunction: 1.92 (Highly right-skewed)
Age: 1.13 (Highly right-skewed)


## 6. Bivariate Analysis: Feature Distributions by Diabetic Outcome

In [7]:
grouped = df.groupby('Outcome').mean().T
grouped.columns = ['Non-Diabetic (0)', 'Diabetic (1)']
grouped['Difference (%)'] = ((grouped['Diabetic (1)'] - grouped['Non-Diabetic (0)']) / grouped['Non-Diabetic (0)'] * 100).round(2)
print('Comparison of Feature Means by Outcome:')
print(grouped.to_string())

Comparison of Feature Means by Outcome:
Feature                   Non-Diabetic (0)  Diabetic (1)  Difference (%)
Pregnancies                           3.30          4.87           47.58
Glucose                             109.98        141.26           28.44
BloodPressure                        68.18         70.82            3.87
SkinThickness                        19.66         22.16           12.72
Insulin                              68.79        100.34           45.86
BMI                                  30.30         35.14           15.97
DiabetesPedigreeFunction              0.43          0.55           27.91
Age                                  31.19         37.07           18.84


## 7. Outlier Detection using Interquartile Range (IQR)

We calculate outliers using the standard $1.5 \times \text{IQR}$ rule. For clinical attributes with missing zero placeholders, we evaluate the valid positive observations ($>0$) to prevent placeholder zeros from corrupting the true quartile bounds.

In [8]:
outlier_rows = []
for col in df.drop(columns=['Outcome']).columns:
    data = df[col][df[col] > 0] if col in quirk_cols else df[col]
    q25 = data.quantile(0.25)
    q75 = data.quantile(0.75)
    iqr = q75 - q25
    lower = q25 - 1.5 * iqr
    upper = q75 + 1.5 * iqr
    n_outliers = ((data < lower) | (data > upper)).sum()
    pct_outliers = (n_outliers / len(data)) * 100
    outlier_rows.append({
        'Feature': col,
        'Valid Rows': len(data),
        'IQR': round(iqr, 2),
        'Lower Bound': round(lower, 2),
        'Upper Bound': round(upper, 2),
        'Outlier Count': n_outliers,
        'Outlier Pct (%)': round(pct_outliers, 2)
    })

outlier_df = pd.DataFrame(outlier_rows)
print('Outlier Detection Summary (1.5 * IQR Rule on Valid Measurements):')
print(outlier_df.to_string(index=False))

Outlier Detection Summary (1.5 * IQR Rule on Valid Measurements):
Feature                   Valid Rows  IQR    Lower Bound  Upper Bound  Outlier Count  Outlier Pct (%)
Pregnancies                      768   5.00        -6.50        13.50              4             0.52
Glucose                          763  42.00        36.00       204.00              0             0.00
BloodPressure                    733  16.00        40.00       104.00             14             1.91
SkinThickness                    541  14.00         1.00        57.00              3             0.55
Insulin                          394 113.75       -94.38       360.62             24             6.09
BMI                              757   9.10        13.85        50.25              8             1.06
DiabetesPedigreeFunction         768   0.38        -0.33         1.20             29             3.78
Age                              768  17.00        -1.50        66.50              9             1.17


## 8. Correlation Matrix & Multicollinearity

In [9]:
corr_matrix = df.corr()
print('Correlation Matrix (features with Outcome):')
print(corr_matrix['Outcome'].sort_values(ascending=False))
print('\nNotable Inter-Feature Correlations:')
print('- Age vs Pregnancies: r = 0.54 (Older women have higher pregnancy counts)')
print('- SkinThickness vs BMI: r = 0.39 (Physical body composition association)')
print('- Glucose vs Insulin: r = 0.33 (Endocrine relationship)')

Correlation Matrix (features with Outcome):
Outcome                     1.000000
Glucose                     0.466581
BMI                         0.292695
Age                         0.238356
Pregnancies                 0.221898
DiabetesPedigreeFunction    0.173844
Insulin                     0.130548
SkinThickness               0.074752
BloodPressure               0.065068
Name: Outcome, dtype: float64

Notable Inter-Feature Correlations:
- Age vs Pregnancies: r = 0.54 (Older women have higher pregnancy counts)
- SkinThickness vs BMI: r = 0.39 (Physical body composition association)
- Glucose vs Insulin: r = 0.33 (Endocrine relationship)


## 9. Comprehensive EDA Summary & Blueprint for Phase 2

### A. Key Predictive Drivers
1. **Glucose:** Strongest individual predictor ($r \approx 0.49$). Diabetic subjects average $141.3$ mg/dL vs $110.0$ mg/dL for non-diabetics (+28.4%).
2. **BMI:** Second strongest predictor ($r \approx 0.31$). Diabetic cohort averages BMI of $35.1$ (obese) vs $30.3$ for non-diabetic cohort (+16.0%).
3. **Insulin & SkinThickness:** Strong predictors once zero-placeholders are masked ($r \approx 0.30$ and $r \approx 0.26$).
4. **Age & Pregnancies:** Noticeable positive association with risk (+18.8% higher age, +47.6% higher pregnancy count in positive cases).

### B. Data Quality & Missingness Challenges
1. **Zero-as-Missing:** `Insulin` is missing in 48.7% of subjects; `SkinThickness` is missing in 29.6%; `BloodPressure` (4.56%), `BMI` (1.43%), and `Glucose` (0.65%) have smaller missingness.
2. **Imputation Strategy for Phase 2:** Because `Insulin` and `SkinThickness` are missing at substantial rates and are non-normally distributed, median imputation or iterative/KNN imputation grouped by diabetic risk or age/BMI bins is strongly indicated over simple mean imputation.

### C. Outliers & Distribution Skewness
1. `Insulin` (24 outliers, 6.09%) and `DiabetesPedigreeFunction` (29 outliers, 3.78%) exhibit high right-skewness ($> 1.9$).
2. Robust scaling (`RobustScaler`) or clipping/winsorizing is a prime candidate during preprocessing to prevent distorted gradients in linear/SVM models.

### D. Feature Engineering Opportunities for Phase 2
1. **BMI Category:** Categorize into WHO clinical brackets (Underweight, Normal, Overweight, Obese).
2. **Age Group:** Categorize into Young (<30), Middle Age (30-50), Senior (>50).
3. **Risk Factor Score:** Composite clinical index summing binary risk flags (e.g. Glucose > 140, BMI > 30, BP > 80, Age > 45).